# RAG System — Sports Science Q&A
**Deep Learning Project — Centrale Lyon 2025-2026**

This notebook walks through the full RAG pipeline step by step:
1. Data loading & chunking
2. Embedding generation
3. ChromaDB indexing
4. Retrieval
5. Generation with HuggingFace LLMs
6. Evaluation (Precision@3, latency, embedding model comparison, prompt comparison)

In [ ]:
import os
import time
import json
from pathlib import Path
from tqdm import tqdm

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

## Step 1 — Data Preparation

The knowledge base contains 114 documents covering sprint biomechanics, ankle mechanics, plyometrics, nutrition, recovery, and elite athlete profiles (Mbappé, Lamine Yamal, Noah Lyles, Usain Bolt).

We use `RecursiveCharacterTextSplitter` to split documents into overlapping chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_FILE = Path('data/sports_science_corpus.json')
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    raw_corpus = json.load(f)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.create_documents([doc['text'] for doc in raw_corpus])
text_lines = [c.page_content for c in chunks]

print(f'Documents: {len(raw_corpus)}')
print(f'Chunks after split: {len(text_lines)}')
print(f'\nSample chunk:\n{text_lines[0]}')

## Step 2 — Embedding Generation

We use `all-MiniLM-L6-v2` from Sentence Transformers — a 22M parameter BERT-based model producing 384-dimensional normalized vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

def emb_text(text: str):
    return embedding_model.encode([text], normalize_embeddings=True).tolist()[0]

vec = emb_text(text_lines[0])
print(f'Model: {EMBEDDING_MODEL}')
print(f'Vector dimension: {len(vec)}')
print(f'Norm (should be ~1.0): {sum(v**2 for v in vec):.4f}')

## Step 3 — ChromaDB Vector Store

We store all chunk embeddings in a persistent ChromaDB collection using cosine similarity and an HNSW index.

In [ ]:
from chromadb import PersistentClient

client = PersistentClient(path='chroma_db')
COLLECTION_NAME = 'sports_science_rag'

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)

embeddings, ids = [], []
for i, line in enumerate(tqdm(text_lines, desc='Embedding')):
    embeddings.append(emb_text(line))
    ids.append(str(i))

collection.add(documents=text_lines, embeddings=embeddings, ids=ids)
print(f'\n{collection.count()} chunks indexed')

## Step 4 — Retrieval

The retrieval function embeds the query and runs a cosine similarity search against the ChromaDB collection.

In [ ]:
def retrieve(question: str, n_results: int = 3):
    results = collection.query(
        query_embeddings=[emb_text(question)],
        n_results=n_results
    )
    return '\n\n'.join(results['documents'][0]), results['documents'][0]

test_q = 'How does ankle dorsiflexion affect sprint acceleration?'
context, docs = retrieve(test_q)

print(f'Query: {test_q}\n')
for i, doc in enumerate(docs):
    print(f'[{i+1}] {doc[:150]}...')

## Step 5 — Generation

We use `google/flan-t5-base` locally (250M params, seq2seq). With a HuggingFace token, `Mistral-7B-Instruct` is available via the free inference API.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t0 = time.time()
_tok   = AutoTokenizer.from_pretrained('google/flan-t5-base')
_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
print(f'Model loaded in {time.time()-t0:.1f}s')

def generate(prompt: str) -> str:
    enc = _tok(prompt[:900], return_tensors='pt', truncation=True, max_length=512)
    out = _model.generate(**enc, max_new_tokens=150, no_repeat_ngram_size=4)
    return _tok.decode(out[0], skip_special_tokens=True)

## Step 5a — Prompt Templates

We test three prompt strategies:

In [ ]:
PROMPT_XML = """Use the information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.

<context>
{context}
</context>

<question>
{question}
</question>"""

PROMPT_STRUCTURED = """Based on the following excerpts from sports science research, answer the question below.

{context}

{question}"""

PROMPT_MINIMAL = "Answer: {question}\nContext: {context}"

def answer(question: str, template: str) -> tuple:
    ctx, _ = retrieve(question)
    t0 = time.time()
    ans = generate(template.format(context=ctx[:700], question=question))
    return ans, round(time.time() - t0, 2)

test_q = 'What muscles are used in sprint acceleration?'
for name, tmpl in [('XML', PROMPT_XML), ('Structured', PROMPT_STRUCTURED), ('Minimal', PROMPT_MINIMAL)]:
    ans, lat = answer(test_q, tmpl)
    print(f'[{name}] ({lat}s)\n{ans[:200]}\n')

## Step 6 — Evaluation

### 6a — Retrieval: Precision@3

For each test query, we check how many of the top-3 retrieved chunks contain the expected keywords.

In [ ]:
EVAL_SET = [
    ('ankle dorsiflexion sprint performance',  ['ankle', 'dorsiflexion']),
    ('horizontal force acceleration',          ['force', 'horizontal', 'acceleration']),
    ('Mbappe Yamal speed comparison',          ['mbappe', 'yamal']),
    ('reactive strength plyometrics',          ['reactive', 'strength']),
    ('sleep recovery athlete',                 ['sleep', 'recovery']),
    ('protein muscle adaptation',              ['protein', 'muscle']),
    ('Noah Lyles 100m sprint',                 ['lyles', 'sprint']),
    ('ankle tendon sprint elasticity',         ['tendon', 'ankle']),
]

def precision_at_k(docs, keywords, k=3):
    return sum(1 for d in docs[:k] if any(kw.lower() in d.lower() for kw in keywords)) / k

precisions, latencies = [], []
print(f'{"Query":<45} {"P@3":>5} {"Latency":>9}')
print('-' * 62)

for query, kws in EVAL_SET:
    t0 = time.time()
    _, docs = retrieve(query)
    lat = (time.time() - t0) * 1000
    p = precision_at_k(docs, kws)
    precisions.append(p)
    latencies.append(lat)
    print(f'{"✓" if p > 0 else "✗"} {query:<44} {p:>5.2f} {lat:>7.1f}ms')

print(f'\nAvg Precision@3 : {sum(precisions)/len(precisions):.3f}')
print(f'Avg Latency     : {sum(latencies)/len(latencies):.2f}ms')

### 6b — Embedding Model Comparison

In [ ]:
MODELS = {
    'all-MiniLM-L6-v2': 'all-MiniLM-L6-v2',
    'paraphrase-multilingual-MiniLM-L12-v2': 'paraphrase-multilingual-MiniLM-L12-v2',
}

for model_name, model_id in MODELS.items():
    m = SentenceTransformer(model_id)
    col_name = f'eval_{model_name[:20].replace("-","_")}'
    try:
        client.delete_collection(col_name)
    except Exception:
        pass
    col2 = client.create_collection(name=col_name, metadata={'hnsw:space': 'cosine'})
    all_embs = m.encode(text_lines, normalize_embeddings=True).tolist()
    col2.add(documents=text_lines, embeddings=all_embs, ids=[str(i) for i in range(len(text_lines))])

    ps = []
    for query, kws in EVAL_SET[:4]:
        qv = m.encode([query], normalize_embeddings=True).tolist()[0]
        res = col2.query(query_embeddings=[qv], n_results=3)
        ps.append(precision_at_k(res['documents'][0], kws))

    print(f'Model: {model_name}')
    print(f'  P@3: {sum(ps)/len(ps):.3f} | Dim: {len(all_embs[0])}\n')

### 6c — End-to-End Demo

In [ ]:
questions = [
    'How does ankle dorsiflexion affect sprint acceleration?',
    'Who is faster between Mbappé and Lamine Yamal?',
    'What exercises improve reactive strength for sprinters?',
    'How does sleep quality affect athletic recovery?',
]

for q in questions:
    ans, lat = answer(q, PROMPT_STRUCTURED)
    print(f'Q: {q}')
    print(f'A: {ans}')
    print(f'   ({lat}s)\n')